# Testing the salary model prediction from the first part of chapter 1

## Set up and load data and models

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [2]:
import pandas as pd

from src.job_intel.models.salary_predictor import predict_salary
from src.job_intel.config import CH1_PROCESSED_SALARY_MODEL_PCA_DF

df = pd.read_csv(CH1_PROCESSED_SALARY_MODEL_PCA_DF)

df.shape, df.head()


((6161, 138),
                                      Job Description  Rating  \
 0  ABOUT HOPPER\n\nAt Hopper, we’re on a mission ...     3.5   
 1  At Noom, we use scientifically proven methods ...     4.5   
 2  Decode_M\n\nhttps://www.decode-m.com/\n\nData ...     NaN   
 3  Sapphire Digital seeks a dynamic and driven mi...     3.4   
 4  Director, Data Science - (200537)\nDescription...     3.4   
 
                      Size  Founded                   Industry  \
 0   501 to 1000 employees   2007.0            Travel Agencies   
 1  1001 to 5000 employees   2008.0  Health, Beauty, & Fitness   
 2       1 to 50 employees      NaN                    Unknown   
 3    201 to 500 employees   2019.0                   Internet   
 4     51 to 200 employees   2007.0    Advertising & Marketing   
 
                    Sector     role_source state ownership_clean  \
 0        Travel & Tourism  data_scientist    NY         private   
 1       Consumer Services  data_scientist    NY         pri

## Test 1 - Predict salary for a fre real jobs

In [25]:
test_rows = df.sample(10, random_state=42)

predictions = []
for idx, row in test_rows.iterrows():
    record = row.to_frame().T
    pred = predict_salary(record)
    predictions.append((idx, pred))

predictions


[(1056, 107763.3515625),
 (410, 96872.265625),
 (3948, 79864.6171875),
 (3821, 80770.3984375),
 (969, 76236.21875),
 (5746, 71101.9296875),
 (4084, 75105.7734375),
 (4439, 100361.375),
 (5684, 70041.9140625),
 (1499, 96529.546875)]

In [30]:
pred = pd.DataFrame(predictions).set_index(0)
pred_comp = pd.DataFrame({'observed' : test_rows['sal_mean'], 'predicted' : pred.loc[:,1]})
pred_comp['diff'] = pred_comp['observed'] - pred_comp['predicted']

print(f" Mean diff {pred_comp['diff'].abs().mean()}"), pred_comp

 Mean diff 23337.615625


(None,
       observed      predicted          diff
 1056  132500.0  107763.351562  24736.648438
 410   146500.0   96872.265625  49627.734375
 3948   66500.0   79864.617188 -13364.617188
 3821   73000.0   80770.398438  -7770.398438
 969    48500.0   76236.218750 -27736.218750
 5746   64500.0   71101.929688  -6601.929688
 4084   59500.0   75105.773438 -15605.773438
 4439   79000.0  100361.375000 -21361.375000
 5684   40000.0   70041.914062 -30041.914062
 1499   60000.0   96529.546875 -36529.546875)

## Test 2 - Mock user input

In [5]:
# Use one existing row to extract valid categorical codes
base = df.iloc[[0]]  # keep as DataFrame

# Start from its categorical values
mock = base.copy()

# Zero all skills
for col in mock.columns:
    if "core_programming" in col or \
       "data_engineering_pipelines" in col or \
       "ml_ai" in col or \
       "analytics_stats" in col or \
       "bi_viz" in col or \
       "cloud" in col or \
       "db_storage" in col or \
       "productivity_workflow" in col or \
       "soft_skills" in col or \
       "domain_specific" in col:
        mock[col] = 0

# Add some custom mock user skills
mock["core_programming__basic"] = 1
mock["core_programming__intermediate"] = 1
mock["ml_ai__basic"] = 1
mock["analytics_stats__basic"] = 1

# Predict salary for the mock profile
mock_pred = predict_salary(mock)

mock_pred


125635.1796875